In [62]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [63]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.rwf2000 import RWF2000Dataset
from src.baseline_cnn_lstm import BaselineCNNLSTM
from src.config import DATASET_ROOT
from src.config import CHECKPOINT_DIR
from tqdm.notebook import tqdm

In [64]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print(device)

cuda:2


In [71]:
hyperparameters = {
    "num_frames": 32, 
    "batch_size": 4,
    "hidden_size": 256,
    "learning_rate": 5e-5,
    "epochs": 15,
    "dropout": 0.4,
}

In [ ]:
# dataset, loaders, model, optimizer, criterion

train_dataset = RWF2000Dataset(DATASET_ROOT, split="train", num_frames=hyperparameters["num_frames"])
val_dataset = RWF2000Dataset(DATASET_ROOT, split="val", num_frames=hyperparameters["num_frames"])

train_loader = DataLoader(
    train_dataset,
    batch_size=hyperparameters["batch_size"],
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=hyperparameters["batch_size"],
    shuffle=False,
    num_workers=4
)

model = BaselineCNNLSTM(
    hidden_size=hyperparameters["hidden_size"],
    num_layers=1,
    num_classes=2,
    dropout=hyperparameters["dropout"],
    freeze_cnn=True
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=hyperparameters["learning_rate"]
)

In [67]:
# training function

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Training", leave=False)
    
    for videos, labels in pbar:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        outputs = model(videos)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * videos.size(0)
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct/total:.4f}"
        )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [68]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc="Validation", leave=False)
    
    with torch.no_grad():
        for videos, labels in pbar:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * videos.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                acc=f"{correct/total:.4f}"
            )
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [69]:
model_name = "baseline_cnn_lstm_v1_unfreeze_5_layers.pt"
save_dir = CHECKPOINT_DIR / "baseline_cnn_lstm" / model_name

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

num_epochs = hyperparameters["epochs"]

best_val_acc = 0.0
epochs_without_improvement = 0
early_stopping_patience = 7

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

for epoch in range(num_epochs):

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)


    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)    
    history["val_acc"].append(val_acc)
    
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "config": hyperparameters,
            "best_val_acc": best_val_acc
        }

        torch.save(checkpoint, save_dir)
        print(f"New best model saved. Val Acc: {best_val_acc:.4f}")

    else:
        epochs_without_improvement += 1
        print(
            f"No improvement for {epochs_without_improvement}/"
            f"{early_stopping_patience} epochs"
        )
        
    if epochs_without_improvement >= early_stopping_patience:
        print("Early stopping triggered.")
        break


Epoch 1/15
--------------------------------------------------


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Train Loss: 0.6336 | Train Acc: 0.6288 | Val Loss: 0.5249 | Val Acc: 0.7700
New best model saved. Val Acc: 0.7700

Epoch 2/15
--------------------------------------------------


Training:   0%|          | 0/400 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Train Loss: 0.5040 | Train Acc: 0.7556 | Val Loss: 0.5183 | Val Acc: 0.7400
No improvement for 4/5 epochs

Epoch 6/15
--------------------------------------------------


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Train Loss: 0.5195 | Train Acc: 0.7244 | Val Loss: 0.5543 | Val Acc: 0.7100
No improvement for 5/5 epochs
Early stopping triggered.
